# 05 – Open question: a small world of actors

**Question.** Is the world of IMDb actors a "small world" (the "six degrees
of separation" made famous by the Kevin Bacon game), who is its real
centre, and how does this change from one decade to the next?

**Prerequisites**: the silver tables (notebook 01), then, inside the Spark container:

```bash
docker compose exec spark python jobs/open_question.py        # about 10 minutes
```

Code: `src/imdb_project/open_question.py`, `jobs/open_question.py`.

In [1]:
import pandas as pd
from pyspark.sql import functions as F

from imdb_project.spark import get_spark

spark = get_spark("notebook-open-question")
spark.sparkContext.setLogLevel("ERROR")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

from imdb_project import open_question as oq
from imdb_project.config import RESULTS_DIR


def result(name):
    return spark.read.parquet(str(RESULTS_DIR / name / "parquet"))

## 1. The co-star graph

- **Vertices**: actors and actresses (`title.principals`, categories
  `actor` / `actress`).
- **Edges**: two people are linked when they appear in the same feature
  film with at least 1,000 votes (same definitions as in the exploration).
- **Distance**: number of steps of the shortest chain actor -> film ->
  actor -> ... (two co-stars are at distance 1).
- Graphs: all years together, then one graph per decade since 1980 built
  with the films of that decade only.

**Limit.** `title.principals` lists the first credited people of each title
(at most 10 actors per film): the graph links billed cast members, not full
casts. Distances are therefore longer than with complete casts (as used by
the "Oracle of Bacon" website).

## 2. Breadth-first search with DataFrames

All the distances come from breadth-first searches (BFS) written with
DataFrame joins:

1. the frontier (actors reached at the previous level) is joined with the
   edges;
2. the actors already visited are removed with an anti-join;
3. the new level is written to Parquet and read back.

Writing each level breaks the Spark lineage: without it, the plan of the
query grows at every level and becomes slower and slower. Several BFS run at
the same time (multi-source BFS): each row carries its source.

GraphFrames was not used: its shortest paths are computed in the same way
(iterated joins), but writing the BFS gives control over the stop condition,
the lineage and the reconstruction of the chain of films.

## 3. Finding the real centre

The centre is the actor with the smallest average distance to all the
other actors of the giant connected component (closeness centrality).
Computing it exactly for everybody would need about 10^10 distances.

1. **Giant component**: BFS from the actor with the most co-stars.
2. **Estimate** (Eppstein & Wang, 2004): BFS from 100 random actors of the
   component (fixed seed, drawn from a hash of the actor id so that the
   sample is the same on every run); the average distance of each actor to
   these 100 estimates its closeness. The same distances give the average
   separation of the graph.
3. **Exact**: BFS from the 10 best estimated actors, the 10 actors with the
   most co-stars and Kevin Bacon; the centre is the one with the smallest
   exact average distance. With a sample of 30 actors, the estimate alone
   could miss the real centre: the most connected actors are a safety net.

In [2]:
summary = result("open_graph_summary").orderBy(F.col("graph") != "all years", "graph")
summary.drop("graph_order", "centre_nconst").toPandas()

,graph,actors,edges,giant_component,giant_component_pct,avg_separation,median_separation,max_distance_seen,pct_pairs_within_6,centre_name,centre_avg_distance,centre_degree,bacon_avg_distance,bacon_rank
0,all years,184397,1950777,177572,96.30,4.558,4.0,12,97.14,Willem Dafoe,3.023,798,3.262,16
1,1980-1989,19217,143270,17627,91.73,4.954,5.0,17,86.20,Harry Dean Stanton,3.344,149,3.732,15
2,1990-1999,22411,182725,20573,91.80,4.819,5.0,13,88.03,Samuel L. Jackson,3.163,239,3.354,19
3,2000-2009,42594,344222,39902,93.68,4.853,5.0,11,91.66,Willem Dafoe,3.216,243,3.564,13
4,2010-2019,70341,570255,66377,94.36,4.878,5.0,14,93.66,Ben Kingsley,3.315,235,3.702,18
5,2020-2025,55206,387865,51102,92.57,5.503,5.0,13,78.49,John Cena,3.797,149,4.469,12


In [3]:
closeness = result("open_closeness")
closeness.orderBy(F.col("graph") != "all years", "graph", "avg_distance").toPandas()

,graph,nconst,primaryName,degree,avg_distance,eccentricity,reached,estimated,top_degree
0,all years,nm0000353,Willem Dafoe,798,3.023,8,177571,True,True
1,all years,nm0000134,Robert De Niro,771,3.044,8,177571,True,True
2,all years,nm0000168,Samuel L. Jackson,826,3.050,8,177571,False,True
3,all years,nm0001426,Ben Kingsley,602,3.057,8,177571,True,False
4,all years,nm0000323,Michael Caine,858,3.063,8,177571,False,True
...,...,...,...,...,...,...,...,...,...
108,2020-2025,nm9627615,Srikanth Iyengar,260,4.629,10,51101,False,True
109,2020-2025,nm3808948,Vennela Kishore,335,4.654,10,51101,False,True
110,2020-2025,nm0103976,Brahmaji,266,4.720,10,51101,False,True
111,2020-2025,nm1960306,Rajeev Kanakala,258,4.809,10,51101,False,True


In [4]:
# Number of actors at each distance from the centre and from Kevin Bacon (all years).
(result("open_distance_distribution").where("graph = 'all years'")
 .groupBy("distance").pivot("primaryName").agg(F.first("count"))
 .orderBy("distance").toPandas())

,distance,Kevin Bacon,Willem Dafoe
0,0,1,1
1,1,451,798
2,2,23407,38106
3,3,91967,99441
4,4,53555,35113
5,5,7434,3736
6,6,640,312
7,7,92,56
8,8,25,9


## 4. Chains of films

In [5]:
for row in result("open_example_paths").collect():
    print(f"distance {row['distance']}: {row['path']}")

distance 2: Kevin Bacon -[National Lampoon's Animal House (1978)]- John Belushi -[1941 (1979)]- Toshirō Mifune
distance 2: Kevin Bacon -[Black Mass (2015)]- Johnny Depp -[Once Upon a Time in Mexico (2003)]- Willem Dafoe
distance 2: Kevin Bacon -[Black Mass (2015)]- Joel Edgerton -[The Great Gatsby (2013)]- Amitabh Bachchan


Any chain can be computed with the `path` step, for instance between Kevin
Bacon (`nm0000102`) and Amitabh Bachchan (`nm0000821`):

```bash
docker compose exec spark python jobs/open_question.py path nm0000102 nm0000821
```

In [6]:
spark.stop()